<a href="https://colab.research.google.com/github/meyhh-lang/Aplikasi-Login/blob/main/%5BKlasifikasi%5D_Submission_Akhir_BMLP_Connary_Zahra_Rameyzanawa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

In [7]:
# 1. Memuat Dataset
data_class = pd.read_csv("data_clustering_inverse (2).csv")

# PENTING: Mengubah tipe data 'Target' menjadi integer agar terbaca sebagai kelas/kategori
data_class["Target"] = data_class["Target"].astype(int)

# Encode kembali kolom kategorikal jika ada
label_encoders = {}
for col in data_class.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    data_class[col] = le.fit_transform(data_class[col].astype(str))
    label_encoders[col] = le

# Pemisahan Fitur (X) dan Target (y)
X = data_class.drop(columns=["Target"])
y = data_class["Target"]

# 2. Pembagian Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
joblib.dump(dt_model, "decision_tree_model.h5")

# 4. Random Forest (Model Eksplorasi)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, "explore_RandomForest_classification")

# 5. Evaluasi Model
models = {"Decision Tree": dt_model, "Random Forest": rf_model}

print("=== EVALUASI MODEL KLASIFIKASI ===")
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n--- {name} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(
        f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}"
    )
    print(f"Recall   : {recall_score(y_test, y_pred, average='weighted'):.4f}")
    print(f"F1-Score : {f1_score(y_test, y_pred, average='weighted'):.4f}")

# 6. Hyperparameter Tuning
param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"],
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1_weighted",
)
grid_search.fit(X_train, y_train)

best_tuning_model = grid_search.best_estimator_
joblib.dump(best_tuning_model, "tuning_classification")

print("\n=== HASIL HYPERPARAMETER TUNING ===")
print("Parameter Terbaik:", grid_search.best_params_)
y_pred_tune = best_tuning_model.predict(X_test)
print("\nClassification Report Model Tuning:\n")
print(classification_report(y_test, y_pred_tune))

=== EVALUASI MODEL KLASIFIKASI ===

--- Decision Tree ---
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000

--- Random Forest ---
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1-Score : 1.0000

=== HASIL HYPERPARAMETER TUNING ===
Parameter Terbaik: {'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 2}

Classification Report Model Tuning:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        99
           1       1.00      1.00      1.00       192
           2       1.00      1.00      1.00        91

    accuracy                           1.00       382
   macro avg       1.00      1.00      1.00       382
weighted avg       1.00      1.00      1.00       382

